# Task 01 — Feature engineering & cleaning

## 🌌 Macrocosm photo-z — outlier study (tasks 2026-06-16)

**✅ GIVEN (do NOT re-derive this):** we already ran a **400k 5-fold cross-validation** with three
models (HGB, RF, MLP) and collected the galaxies that *all three* predict catastrophically out-of-fold
(|Δz/(1+z)| > 0.05). There are **6,974** such **hard** galaxies. Their objids are in
**`../hard_objids.csv`**. Treat this hard set as a **fixed input** — your job is to characterize / act
on it, not to re-find it from 400k.

**Catalog:** `gs://macrocosm-lewagon/data/sample_v1/catalog_v1.parquet` (600k rows, 55 columns).
The SDSS **`-9999` sentinel** means "not measured" — always clean it to NaN first.

**Metric:** `σ_MAD = 1.4826 · median(|Δz − median(Δz)|)`, with `Δz = (z_pred − z_true)/(1+z)`;
an **outlier** is `|Δz| > 0.05`. σ_MAD is the headline metric, report **outlier rate** alongside it.

---

Build the modelling feature set from the raw catalog and sanity-check it. Everyone else builds on this
recipe, so get it right: clean the `-9999` sentinel, derive the 4 colors, log the right-skewed sizes.

In [4]:
# === shared setup: load catalog, clean -9999, build the 16 features ===
import pandas as pd, numpy as np

CATALOG = "/Users/cathyzhou/code/Le-Wagon-Macrocosm/Macrocosm/notebooks/tasks-2026-6-16/01-feature-engineering/data_sample_v4.5_catalog_v4.parquet"
# Colab: from google.colab import auth; auth.authenticate_user()
# or download once: !gcloud storage cp $CATALOG . , then CATALOG = "catalog_v1.parquet"

FEATS = ["dered_u","dered_g","dered_r","dered_i","dered_z",
         "log_expRad_r","log_deVRad_r","log_petroRad_r","log_petroR50_r","log_petroR90_r",
         "fracDeV_r", "ra", "dec"]

def load_features(path=CATALOG, n=None, seed=0):
    """Load catalog, clean the -9999 sentinel, build colors / log-sizes / conc.
    Returns (D, cat): D = features+redshift+objid (optionally subsampled), cat = full cleaned catalog."""
    cat = pd.read_parquet(path)

    num = cat.select_dtypes("number").columns
    cat[num] = cat[num].mask(cat[num] <= -100)                       # clean SDSS -9999
    for a, b in [("u","g"),("g","r"),("r","i"),("i","z")]:
        cat[f"{a}-{b}"] = (cat[f"dered_{a}"] - cat[f"dered_{b}"]).clip(-1, 4)
    for s in ["expRad_r","deVRad_r","petroRad_r","petroR50_r","petroR90_r"]:
        cat["log_"+s] = np.log1p(cat[s].clip(lower=0))
    cat["conc_r"] = cat["petroR90_r"] / cat["petroR50_r"].replace(0, np.nan)
    D = cat[["objid", "redshift"] + FEATS].replace([np.inf,-np.inf], np.nan).dropna()
    if n:
        D = D.sample(n, random_state=seed).reset_index(drop=True)
    return D, cat

def smad(dz): return 1.4826 * np.median(np.abs(dz - np.median(dz)))

def metrics(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    dz = (y_pred - y_true) / (1 + y_true)
    return {"MAE": round(float(np.mean(np.abs(y_pred-y_true))), 5),
            "sigma_MAD": round(float(smad(dz)), 5),
            "outlier_rate": round(float(np.mean(np.abs(dz) > 0.05)), 5)}

# the GIVEN hard set (6,974 objids from the 400k 5-fold CV)
HARD = set(pd.read_csv("/Users/cathyzhou/code/Le-Wagon-Macrocosm/Macrocosm/notebooks/tasks-2026-6-16/hard_objids.csv")["objid"])
print(f"hard set: {len(HARD)} galaxies")

hard set: 6974 galaxies


❓ **Question (load & clean)** ❓

👇 Load 100k rows with `load_features(n=100000, seed=0)`. Print `D.shape` and `FEATS`.

In [5]:
D, cat = load_features(n=93026, seed=0)

print(D.shape)
print(FEATS)

(93026, 15)
['dered_u', 'dered_g', 'dered_r', 'dered_i', 'dered_z', 'log_expRad_r', 'log_deVRad_r', 'log_petroRad_r', 'log_petroR50_r', 'log_petroR90_r', 'fracDeV_r', 'ra', 'dec']


In [ ]:
for a in D.columns:
    if a[0:3] =='log':
        D = D.rename(columns={a:a[4:]})

D.head()


log
log
log
log
log


,objid,redshift,dered_u,dered_g,dered_r,dered_i,dered_z,expRad_r,deVRad_r,petroRad_r,petroR50_r,petroR90_r,fracDeV_r,ra,dec
0,1237662266457194520,0.112145,18.82056,17.69047,17.41725,16.97401,16.89037,1.112416,1.693325,1.784529,1.185122,1.842179,0.315608,213.462014,5.636424
1,1237661384384643273,0.276008,23.02207,19.67287,18.16807,17.61137,17.25266,0.795538,1.163274,1.756761,1.079071,1.919783,1.000000,149.431024,37.621904
2,1237655504035184915,0.023889,16.59799,15.05572,14.33840,13.96385,13.66651,1.913169,2.071690,2.560459,1.815399,2.543352,0.727834,257.383676,34.427037
3,1237662193455923265,0.109475,18.66995,17.19984,16.46021,16.06367,15.82864,1.855604,2.729544,2.287262,1.652192,2.267817,0.025088,194.623387,41.075242
4,1237657775001501938,0.110181,19.78686,17.85513,16.87250,16.47241,16.11247,0.908372,1.127324,1.652919,0.996888,1.842504,1.000000,125.237386,33.099529


In [82]:
head_c4_data = D.round(2).head()
head_c4_data

,objid,redshift,dered_u,dered_g,dered_r,dered_i,dered_z,expRad_r,deVRad_r,petroRad_r,petroR50_r,petroR90_r,fracDeV_r,ra,dec
0,1237662266457194520,0.11,18.82,17.69,17.42,16.97,16.89,1.11,1.69,1.78,1.19,1.84,0.32,213.46,5.64
1,1237661384384643273,0.28,23.02,19.67,18.17,17.61,17.25,0.80,1.16,1.76,1.08,1.92,1.00,149.43,37.62
2,1237655504035184915,0.02,16.60,15.06,14.34,13.96,13.67,1.91,2.07,2.56,1.82,2.54,0.73,257.38,34.43
3,1237662193455923265,0.11,18.67,17.20,16.46,16.06,15.83,1.86,2.73,2.29,1.65,2.27,0.03,194.62,41.08
4,1237657775001501938,0.11,19.79,17.86,16.87,16.47,16.11,0.91,1.13,1.65,1.00,1.84,1.00,125.24,33.10


In [83]:
tail_c4_data = D.round(2).tail()
tail_c4_data

,objid,redshift,dered_u,dered_g,dered_r,dered_i,dered_z,expRad_r,deVRad_r,petroRad_r,petroR50_r,petroR90_r,fracDeV_r,ra,dec
93021,1237679997700079899,0.29,21.64,20.32,18.66,18.08,17.67,0.67,0.87,1.40,0.84,1.48,0.73,347.12,-2.19
93022,1237665329866277133,0.26,23.61,20.29,18.79,18.15,17.78,0.80,0.99,1.55,1.04,1.79,1.00,221.09,26.81
93023,1237680529741447603,0.22,20.87,18.87,17.48,16.97,16.60,0.72,1.12,1.82,1.14,2.01,1.00,346.59,28.60
93024,1237659161204883931,0.30,21.53,19.54,17.98,17.39,17.05,0.68,1.02,1.93,1.18,2.14,1.00,247.88,33.75
93025,1237666275810279595,0.08,18.75,16.99,16.15,15.76,15.44,1.10,1.45,2.13,1.32,2.30,1.00,15.66,25.13


In [84]:
sample_galaxy_df = pd.concat([head_c4_data, tail_c4_data])
sample_galaxy_df

,objid,redshift,dered_u,dered_g,dered_r,dered_i,dered_z,expRad_r,deVRad_r,petroRad_r,petroR50_r,petroR90_r,fracDeV_r,ra,dec
0,1237662266457194520,0.11,18.82,17.69,17.42,16.97,16.89,1.11,1.69,1.78,1.19,1.84,0.32,213.46,5.64
1,1237661384384643273,0.28,23.02,19.67,18.17,17.61,17.25,0.80,1.16,1.76,1.08,1.92,1.00,149.43,37.62
2,1237655504035184915,0.02,16.60,15.06,14.34,13.96,13.67,1.91,2.07,2.56,1.82,2.54,0.73,257.38,34.43
3,1237662193455923265,0.11,18.67,17.20,16.46,16.06,15.83,1.86,2.73,2.29,1.65,2.27,0.03,194.62,41.08
4,1237657775001501938,0.11,19.79,17.86,16.87,16.47,16.11,0.91,1.13,1.65,1.00,1.84,1.00,125.24,33.10
93021,1237679997700079899,0.29,21.64,20.32,18.66,18.08,17.67,0.67,0.87,1.40,0.84,1.48,0.73,347.12,-2.19
93022,1237665329866277133,0.26,23.61,20.29,18.79,18.15,17.78,0.80,0.99,1.55,1.04,1.79,1.00,221.09,26.81
93023,1237680529741447603,0.22,20.87,18.87,17.48,16.97,16.60,0.72,1.12,1.82,1.14,2.01,1.00,346.59,28.60
93024,1237659161204883931,0.30,21.53,19.54,17.98,17.39,17.05,0.68,1.02,1.93,1.18,2.14,1.00,247.88,33.75
93025,1237666275810279595,0.08,18.75,16.99,16.15,15.76,15.44,1.10,1.45,2.13,1.32,2.30,1.00,15.66,25.13


In [6]:
random_galaxy = D.sample(n=6)
random_galaxy

,objid,redshift,dered_u,dered_g,dered_r,dered_i,dered_z,log_expRad_r,log_deVRad_r,log_petroRad_r,log_petroR50_r,log_petroR90_r,fracDeV_r,ra,dec
77586,1237680092724986167,0.154666,20.15192,18.27258,17.16899,16.70268,16.34152,0.825869,1.151388,1.635435,1.039665,1.895317,1.000000,341.589438,16.701892
67512,1237664835927408939,0.052143,19.25814,18.12436,17.64819,17.41210,17.20523,1.211134,1.784693,1.764774,1.187617,1.870840,0.337726,129.479718,23.227498
3771,1237679340027773067,0.190649,21.31028,18.85825,17.59178,17.09097,16.74483,0.695846,1.014045,1.621880,1.004059,1.850796,1.000000,25.415478,-7.223558
83456,1237655498675126633,0.233940,21.15029,19.49577,18.06934,17.57344,17.16588,0.829283,1.195367,1.684517,1.062628,1.803843,1.000000,228.126637,-1.916191
56617,1237657220411424971,0.131272,20.50123,18.59925,17.58311,17.16290,16.85930,0.711278,0.959312,1.473855,0.911107,1.729234,1.000000,158.206298,53.695364
37042,1237670955178000516,0.127703,18.89721,17.77748,17.22145,16.88940,16.73304,1.484010,2.305605,1.969161,1.394005,1.950085,0.000000,25.308761,-10.359202


In [ ]:
def round_radius_columns(df):
    df = df.copy()
    df = df.round(2)
    radius_cols = [
        c for c in ["expRad_r", "deVRad_r", "devRRad_r", "petroRad_r", "petroR50_r", "petroR90_r"]
        if c in df.columns
    ]
    if radius_cols:
        df.loc[:, radius_cols] = df.loc[:, radius_cols].round(1)
    return df

random_galaxy = round_radius_columns(random_galaxy)

In [80]:
from pathlib import Path
import json

import os

#sample_galaxy = sample_galaxy_df.to_json(orient='records')
random_galaxy_js = random_galaxy.to_json(orient='records')
file = '/Users/cathyzhou/code/Le-Wagon-Macrocosm/Macrocosm/notebooks/sdss_images/sample_galaxy.json'
with open(file, 'w') as f:
    for text in eval(random_galaxy_js):
        f.write(f"{text}\n".replace("'", '"'))

In [29]:
random_galaxy[random_galaxy['objid'] == 1237670955178000516]

,objid,redshift,dered_u,dered_g,dered_r,dered_i,dered_z,log_expRad_r,log_deVRad_r,log_petroRad_r,log_petroR50_r,log_petroR90_r,fracDeV_r,ra,dec
37042,1237670955178000516,0.127703,18.89721,17.77748,17.22145,16.8894,16.73304,1.48401,2.305605,1.969161,1.394005,1.950085,0.0,25.308761,-10.359202


In [ ]:
import numpy as np, warnings
from astroquery.sdss import SDSS
from astropy.coordinates import SkyCoord
from astropy.wcs import WCS, FITSFixedWarning
from astropy.nddata import Cutout2D
import astropy.units as u

warnings.simplefilter('ignore', FITSFixedWarning)   # silence harmless WCS warnings

def cutout(ra, dec, band, size=64):
    # 1. Define sky position
    pos_sky = SkyCoord(ra, dec, unit='deg')
    # 2. Get image from SDSS
    # Note: get_images returns a list of HDULists. We take the first one [0]
    try:
        hdu_list = SDSS.get_images(coordinates=pos_sky, band=band, radius=8*u.arcsec, data_release=17)
    except Exception as e:
        raise ValueError(f"Failed to fetch image from SDSS: {e}")

    if not hdu_list or len(hdu_list) == 0:
        raise ValueError("No images found for these coordinates.")

    hdu = hdu_list[0]
    data = hdu[0].data
    header = hdu[0].header

    # 3. Create WCS object
    w = WCS(header)

    # 4. Convert sky position to pixel coordinates explicitly
    # This returns (x, y) in pixel coordinates
    pix_pos_pre = w.world_to_pixel(pos_sky)
    pix_pos = [x.item() for x in pix_pos_pre]

    # 5. Create the cutout
    # Pass the pixel position directly.
    # Use limit_rounding_method='round' or 'floor'/'ceil' if needed, but explicit pixels usually avoid this.
    try:
        cutout_obj = Cutout2D(
            data=data,
            position=pix_pos,      # Explicit pixel coordinates (x, y)
            size=(size, size),     # Output size in pixels
            wcs=w,                 # Optional: preserves WCS info
            mode='partial',        # Allows partial overlap
            fill_value=0           # Value for pixels outside original image
        )
        return cutout_obj.data.astype('float32')
    except Exception as e:
        # If it still fails, it might be due to the position being completely outside the image
        raise ValueError(f"Failed to create cutout at pixel pos {pix_pos}: {e}")


    #try with 1 galaxy
ra = random_galaxy[random_galaxy['objid']==1237679340027773067]['ra']
dec = random_galaxy[random_galaxy['objid']==1237679340027773067]['dec']

stamp_64 = np.stack([cutout(ra, dec, b) for b in 'ugriz'], axis=-1)
print(stamp_64.shape)

np.save(f"{obj_id}_64.npy", stamp_64)

[275.7421497040634, 687.135344429268]
[272.13511757939204, 690.1487529666512]
[272.57531162904843, 677.444617529717]
[270.633609014752, 680.1689343417048]
[269.18382126601307, 684.8510838399467]
(64, 64, 5)


In [62]:
for obj_id in random_galaxy['objid']:
    ra = random_galaxy[random_galaxy['objid']==obj_id]['ra']
    dec = random_galaxy[random_galaxy['objid']==obj_id]['dec']


    stamp_64 = np.stack([cutout(ra, dec, b) for b in 'ugriz'], axis=-1)   # -> (64, 64, 5)
    print(stamp_64.shape)   # (1, 24, 24, 5)

    np.save(f"{obj_id}_64.npy", stamp_64)

[1355.5290144414855, 768.8732620383706]
[1356.0774881841844, 771.7395724087158]
[1355.1265971803252, 761.1650663447624]
[1356.82970633301, 763.6330607399974]
[1355.7712400104856, 767.577727844554]
(64, 64, 5)
[1884.7071470160406, 480.06212809274905]
[1885.3819932720896, 483.5132606011311]
[1885.3660319475962, 471.80770187685533]
[1884.4782726463582, 474.53299338456145]
[1884.6832583415242, 479.14312249377923]
(64, 64, 5)
[275.7421497040634, 687.135344429268]
[272.13511757939204, 690.1487529666512]
[272.57531162904843, 677.444617529717]
[270.633609014752, 680.1689343417048]
[269.18382126601307, 684.8510838399467]
(64, 64, 5)
[1152.6381647760709, 1203.3951677544871]
[1152.4421256279902, 1206.6497050419423]
[1149.3725707414937, 1197.2641405664522]
[1148.4200302882589, 1199.6945876167672]
[1149.6432946202647, 1203.5150837358258]
(64, 64, 5)
[1047.06519394971, 1039.8081153620305]
[1044.293889361786, 1047.2281053485874]
[1046.9550650265212, 1036.5441114953264]
[1046.592046190018, 1040.609966